# Trading Strategies: Momentum, Mean Reversion & EM FX Carry

This notebook demonstrates three fundamental trading strategies:

1. **Momentum**: Trend-following across rates/futures
2. **Mean Reversion**: Fade extremes and reversals  
3. **EM FX Carry** (NEW): Interest rate differential trades

Each strategy includes:
- Signal generation
- Portfolio construction
- Risk management
- Performance evaluation

In [ ]:
# Setup
import sys
sys.path.insert(0, '..')

import numpy as np
import polars as pl
import matplotlib.pyplot as plt
from datetime import date, timedelta

# Futures signals (existing)
from Signals.Futures.MomentumSignal import MomentumSignal
from Signals.Futures.MeanReversionSignal import MeanReversionSignal

# NEW: EM FX Carry
from Signals.EMFXCarrySignal import EMFXCarrySignal

# Set random seed
np.random.seed(42)

## Part 1: Momentum Strategy

**Concept**: "The trend is your friend"
- Buy winners, sell losers
- 12-month lookback (standard)
- Skip last month to avoid reversals
- Cross-sectional ranking

In [ ]:
# Create momentum signal
momentum = MomentumSignal(
    lookback_days=252,  # ~12 months
    standardize=True     # Z-score normalization
)

# Generate synthetic futures price data
dates = [date(2024, 1, 1) + timedelta(days=i) for i in range(365)]

# Create 5 futures contracts with different trends
futures_prices = {
    "STRONG_TREND": [100 * (1.002 ** i) for i in range(365)],      # Strong uptrend
    "WEAK_TREND": [100 * (1.0005 ** i) for i in range(365)],       # Weak uptrend
    "FLAT": [100 + np.random.randn() for _ in range(365)],         # Range-bound
    "WEAK_DOWN": [100 * (0.9995 ** i) for i in range(365)],        # Weak downtrend
    "STRONG_DOWN": [100 * (0.998 ** i) for i in range(365)],       # Strong downtrend
}

# Create DataFrames
momentum_data = {}
for contract, prices in futures_prices.items():
    momentum_data[contract] = pl.DataFrame({
        "date": dates,
        "price": prices
    })

print("Created futures price data for momentum backtest")
print(f"Contracts: {list(futures_prices.keys())}")
print(f"Date range: {dates[0]} to {dates[-1]}")

In [ ]:
# Calculate momentum signals (simulating calculate() method)
# Note: Real implementation would use MDP, this is simplified

momentum_signals = {}
eval_date = dates[-1]

for contract, data in momentum_data.items():
    # Simple momentum: (current - 252 days ago) / 252 days ago
    recent = data.filter(pl.col("date") == eval_date)
    past = data.filter(pl.col("date") == eval_date - timedelta(days=252))
    
    if recent.height > 0 and past.height > 0:
        current_price = recent["price"][0]
        past_price = past["price"][0]
        momentum_signals[contract] = (current_price - past_price) / past_price

# Z-score normalization
values = np.array(list(momentum_signals.values()))
mean = np.mean(values)
std = np.std(values)
momentum_z = {k: (v - mean) / std for k, v in momentum_signals.items()}

# Display results
print("\nMomentum Signals (Z-Scored):")
print("=" * 50)
for contract in sorted(momentum_z.keys(), key=lambda k: momentum_z[k], reverse=True):
    signal = momentum_z[contract]
    print(f"{contract:20s}: {signal:+.2f} {'[LONG]' if signal > 0.5 else '[SHORT]' if signal < -0.5 else ''}")

## Part 2: Mean Reversion Strategy

**Concept**: "What goes up must come down"
- Fade extremes (high z-scores)
- Bollinger bands
- Statistical arbitrage
- Works well in range-bound markets

In [ ]:
# Create mean reversion signal
mean_rev = MeanReversionSignal(
    lookback_days=20,    # 20-day moving average
    standardize=True
)

## Part 3: EM FX Carry Strategy (NEW CAPABILITY)

**Concept**: "Borrow cheap, lend expensive"
- Borrow in low-yield currency (USD: 5.5%)
- Lend in high-yield EM currency (BRL: 13.75%)
- Profit = interest differential - FX depreciation
- UIP violation: High-yield currencies don't depreciate as much as expected

**Risk**: FX volatility can wipe out carry
**Solution**: Carry-to-risk ratio (carry / FX vol)

In [ ]:
# Create EM FX carry signal
fx_carry = EMFXCarrySignal(
    funding_currency="USD",
    lookback_days=60,
    risk_adjust=True,      # Adjust for FX volatility
    standardize=True       # Use BaseSignal z-score standardization
)

# Define EM currency universe with interest rates (as of example date)
em_currencies = {
    "BRL": {"rate": 0.1375, "name": "Brazilian Real"},        # 13.75%
    "TRY": {"rate": 0.2500, "name": "Turkish Lira"},          # 25.00%
    "ZAR": {"rate": 0.0850, "name": "South African Rand"},    # 8.50%
    "MXN": {"rate": 0.1100, "name": "Mexican Peso"},          # 11.00%
    "RUB": {"rate": 0.1600, "name": "Russian Ruble"},         # 16.00%
    "INR": {"rate": 0.0650, "name": "Indian Rupee"},          # 6.50%
}

usd_rate = 0.055  # 5.5%

# Generate synthetic FX rate data (spot rates vs USD)
dates_fx = [date(2024, 1, 1) + timedelta(days=i) for i in range(90)]

# Create realistic FX paths with different volatilities
fx_data = {}
np.random.seed(42)

for currency, info in em_currencies.items():
    # Higher carry often comes with higher vol
    carry = info["rate"] - usd_rate
    base_vol = 0.10 + carry * 0.5  # Positive correlation between carry and vol
    
    # Generate FX path (geometric Brownian motion)
    initial_rate = {
        "BRL": 5.0,
        "TRY": 32.0,
        "ZAR": 18.5,
        "MXN": 17.0,
        "RUB": 90.0,
        "INR": 83.0
    }[currency]
    
    fx_rates = [initial_rate]
    for _ in range(89):
        # Daily return
        daily_vol = base_vol / np.sqrt(252)
        drift = -carry / 365  # UIP violation: depreciate less than carry suggests
        shock = np.random.randn() * daily_vol
        fx_rates.append(fx_rates[-1] * (1 + drift + shock))
    
    # Create DataFrame
    fx_data[currency] = pl.DataFrame({
        "date": dates_fx,
        "interest_rate": [info["rate"]] * len(dates_fx),
        "fx_rate": fx_rates,
        "usd_rate": [usd_rate] * len(dates_fx)
    })

print("Created EM FX data:")
print("=" * 60)
print(f"{'Currency':<10} {'Name':<25} {'Rate':<10} {'Carry vs USD'}")
print("=" * 60)
for currency, info in em_currencies.items():
    carry = info["rate"] - usd_rate
    print(f"{currency:<10} {info['name']:<25} {info['rate']*100:>5.2f}%     {carry*100:>+6.2f}%")
print("\n" + "=" * 60)
print(f"Funding (USD): {usd_rate*100:.2f}%")
print("=" * 60)

In [ ]:
# Evaluate EM FX carry signals using BaseSignal.generate_batch()

# Convert fx_data dict to list for BaseSignal interface
currencies_list = list(em_currencies.keys())
fx_data_list = [fx_data[curr] for curr in currencies_list]

# Use generate_batch() to get z-scores
z_scores = fx_carry.generate_batch(
    inst_data_list=fx_data_list,
    market_data=None,
    as_of=dates_fx[-1]
)

# Convert z-scores to dict
carry_signals = {currency: float(z_scores[i]) for i, currency in enumerate(currencies_list)}

print("\nEM FX Carry Signals (Risk-Adjusted, Z-Scored):")
print("=" * 70)
print(f"{'Currency':<10} {'Name':<25} {'Raw Carry':<12} {'Signal':<10} {'Action'}")
print("=" * 70)

for currency in sorted(carry_signals.keys(), key=lambda k: carry_signals[k], reverse=True):
    signal = carry_signals[currency]
    raw_carry = (em_currencies[currency]["rate"] - usd_rate) * 100
    
    if signal > 0.5:
        action = "[LONG]"
    elif signal < -0.5:
        action = "[SHORT]"
    else:
        action = ""
    
    print(f"{currency:<10} {em_currencies[currency]['name']:<25} {raw_carry:>5.2f}%       {signal:>+5.2f}      {action}")

print("\nNote: Signals are risk-adjusted (carry / FX volatility)")
print("      High raw carry with high vol may rank lower than moderate carry with low vol")

In [ ]:
# Generate dollar-neutral portfolio
portfolio_weights = fx_carry.generate_portfolio_weights(
    signals=carry_signals,
    equal_weight=False  # Proportional to signal strength
)

print("\nDollar-Neutral FX Carry Portfolio:")
print("=" * 50)
print(f"{'Currency':<10} {'Weight':<10} {'Position'}")
print("=" * 50)

longs_total = sum(w for w in portfolio_weights.values() if w > 0)
shorts_total = sum(w for w in portfolio_weights.values() if w < 0)

for currency in sorted(portfolio_weights.keys(), key=lambda k: portfolio_weights[k], reverse=True):
    weight = portfolio_weights[currency]
    
    if abs(weight) < 0.01:
        continue  # Skip near-zero weights
    
    position = "Long" if weight > 0 else "Short"
    print(f"{currency:<10} {weight:>+6.1%}    {position}")

print("=" * 50)
print(f"Long exposure:  {longs_total:>+6.1%}")
print(f"Short exposure: {shorts_total:>+6.1%}")
print(f"Net exposure:   {sum(portfolio_weights.values()):>+6.1%} (should be ~0)")

# Plot carry vs volatility
fig, ax = plt.subplots(figsize=(10, 6))

carry_values = []
vol_values = []
colors = []

for currency in em_currencies.keys():
    raw_carry = em_currencies[currency]["rate"] - usd_rate
    
    # Calculate realized FX vol
    fx_returns = fx_data[currency]["fx_rate"].to_numpy()
    log_returns = np.diff(np.log(fx_returns))
    vol = np.std(log_returns) * np.sqrt(252)
    
    carry_values.append(raw_carry * 100)
    vol_values.append(vol * 100)
    
    # Color by signal
    signal = carry_signals[currency]
    if signal > 0.5:
        colors.append('green')
    elif signal < -0.5:
        colors.append('red')
    else:
        colors.append('gray')

ax.scatter(vol_values, carry_values, s=200, c=colors, alpha=0.6)

for i, currency in enumerate(em_currencies.keys()):
    ax.annotate(currency, (vol_values[i], carry_values[i]), 
                xytext=(5, 5), textcoords='offset points', fontsize=10)

ax.set_xlabel('FX Volatility (%)', fontsize=12)
ax.set_ylabel('Carry vs USD (%)', fontsize=12)
ax.set_title('EM FX: Carry vs Risk\n(Green=Long, Red=Short, Gray=Neutral)', fontsize=14)
ax.grid(True, alpha=0.3)

# Add diagonal lines for carry-to-risk ratios
x_range = np.array([0, max(vol_values)])
for ratio in [0.5, 1.0, 1.5]:
    ax.plot(x_range, ratio * x_range, '--', alpha=0.3, label=f'C/R = {ratio:.1f}')

ax.legend()
plt.tight_layout()
plt.show()

print("\nInterpretation:")
print("- Points above the line have better carry-to-risk ratios")
print("- Green currencies get allocated to longs")
print("- Red currencies get allocated to shorts")
print("- Gray are neutral (carry doesn't justify the risk)")

## Strategy Comparison

| Strategy | Best For | Risk | Typical Holding |
|----------|----------|------|----------------|
| **Momentum** | Trending markets | Reversal risk | 3-12 months |
| **Mean Reversion** | Range-bound markets | Trend continuation | Days to weeks |
| **EM FX Carry** | Stable macro conditions | FX crashes, policy changes | Months to years |

## Key Takeaways

1. **Momentum**: Works until it doesn't (trend breaks)
2. **Mean Reversion**: Timing is critical (too early = losses)
3. **EM FX Carry**: Earns slowly, loses fast (tail risk)

## Next Steps

- Backtest on real data
- Combine strategies (diversification)
- Add risk management (stop-losses, position sizing)
- Implement in ARBS backtest engine